### Product Demand Forecasting

#### 1. Business Understanding

The PT Frozen Foods platform now incorporates a Machine Learning workflow focused on product demand forecasting.

The objective of this notebook is to analyze historical product sales behavior and prepare the foundation for a forecasting model capable of estimating future product demand.

The forecasting use case supports business decisions related to:

- inventory planning
- purchasing decisions
- supplier planning
- logistics preparation
- commercial strategy
- marketing actions
- financial planning

The selected forecasting target is product quantity sold.

Official forecasting grain:

    data_pedido + produto_id

Official target:

    quantity_sold

Forecast horizon:

    90 days

This notebook is exploratory and is not intended to be used as a production processing artifact.

In [ ]:
# ========================================
# ENVIRONMENT VALIDATION
# ========================================

import sys
import sklearn
import xgboost
import lightgbm
import mlflow

print("=" * 80)
print("ENVIRONMENT VALIDATION")
print("=" * 80)

print(f"Python version:        {sys.version}")
print(f"Spark version:         {spark.version}")
print(f"Scikit-learn version:  {sklearn.__version__}")
print(f"XGBoost version:       {xgboost.__version__}")
print(f"LightGBM version:      {lightgbm.__version__}")
print(f"MLflow version:        {mlflow.__version__}")

print("=" * 80)
print("ENVIRONMENT VALIDATION COMPLETED")
print("=" * 80)

In [ ]:
# ========================================
# 0. CONFIGURATION
# ========================================

import pandas as pd
import numpy as np

import pyspark.sql.functions as F
from pyspark.sql.window import Window

import matplotlib.pyplot as plt
import plotly.express as px

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "gold"
TARGET_SCHEMA = "gold"

DOMAIN = "ml"
USE_CASE = "product_demand_forecasting"

SOURCE_DATASET = "analytics_sales_overview"
SOURCE_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{SOURCE_DATASET}"

FORECAST_GRAIN = [
    "data_pedido",
    "produto_id"
]

TARGET_COLUMN = "quantity_sold"

FORECAST_HORIZON_DAYS = 90

print("=" * 80)
print("STARTING ML EXPLORATORY NOTEBOOK: product_demand_forecasting")
print("=" * 80)

print(f"Source table:          {SOURCE_TABLE}")
print(f"Forecast grain:        {FORECAST_GRAIN}")
print(f"Target column:         {TARGET_COLUMN}")
print(f"Forecast horizon days: {FORECAST_HORIZON_DAYS}")

In [ ]:
# ========================================
# 1. LOAD DATASET
# ========================================

print("=" * 80)
print("LOADING SOURCE DATASET")
print("=" * 80)

source_df = spark.table(SOURCE_TABLE)

print(f"[INFO] Source table loaded successfully: {SOURCE_TABLE}")

print("[INFO] Dataset schema:")
source_df.printSchema()

print("=" * 80)
print("SOURCE DATASET PREVIEW")
print("=" * 80)

display(
    source_df.limit(10)
)

In [ ]:
# ========================================
# 2. INITIAL DATASET VALIDATION
# ========================================

print("=" * 80)
print("INITIAL DATASET VALIDATION")
print("=" * 80)

total_rows = source_df.count()

date_stats = (
    source_df
    .agg(
        F.min("data_pedido").alias("min_data_pedido"),
        F.max("data_pedido").alias("max_data_pedido"),
        F.countDistinct("data_pedido").alias("distinct_order_dates"),
        F.countDistinct("produto_id").alias("distinct_products")
    )
    .collect()[0]
)

grain_rows = (
    source_df
    .select(FORECAST_GRAIN)
    .distinct()
    .count()
)

duplicate_grain_rows = total_rows - grain_rows

null_check = (
    source_df
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in FORECAST_GRAIN + [TARGET_COLUMN]
    ])
    .collect()[0]
    .asDict()
)

print(f"Total rows:              {total_rows:,}")
print(f"Min order date:          {date_stats['min_data_pedido']}")
print(f"Max order date:          {date_stats['max_data_pedido']}")
print(f"Distinct order dates:    {date_stats['distinct_order_dates']:,}")
print(f"Distinct products:       {date_stats['distinct_products']:,}")
print(f"Distinct grain rows:     {grain_rows:,}")
print(f"Duplicate grain rows:    {duplicate_grain_rows:,}")
print(f"Null check:              {null_check}")

if any(value != 0 for value in null_check.values()):
    raise ValueError("Null values found in critical forecasting columns.")

print("[INFO] Initial dataset validation completed.")

In [ ]:
# ========================================
# 3. AGGREGATE FORECASTING DATASET
# ========================================

print("=" * 80)
print("AGGREGATING FORECASTING DATASET")
print("=" * 80)

forecast_base_df = (
    source_df
    .groupBy(
        "data_pedido",
        "produto_id"
    )
    .agg(
        F.sum("quantity_sold").alias("quantity_sold"),
        F.sum("net_sales_amount").alias("net_sales_amount"),
        F.sum("gross_sales_amount").alias("gross_sales_amount"),
        F.sum("total_cost_amount").alias("total_cost_amount"),
        F.sum("gross_margin_amount").alias("gross_margin_amount"),
        F.sum("order_count").alias("order_count"),
        F.sum("line_count").alias("line_count"),

        F.first("produto_nome", ignorenulls=True).alias("produto_nome"),
        F.first("categoria", ignorenulls=True).alias("categoria"),
        F.first("marca", ignorenulls=True).alias("marca"),
        F.first("status_produto", ignorenulls=True).alias("status_produto"),

        F.countDistinct("cliente_id").alias("distinct_customers"),
        F.countDistinct("canal_id").alias("distinct_channels"),
        F.countDistinct("vendedor_id").alias("distinct_salespersons")
    )
)

forecast_base_df = (
    forecast_base_df
    .withColumn("avg_net_unit_price", F.col("net_sales_amount") / F.col("quantity_sold"))
    .withColumn("avg_gross_unit_price", F.col("gross_sales_amount") / F.col("quantity_sold"))
    .withColumn("avg_unit_cost", F.col("total_cost_amount") / F.col("quantity_sold"))
    .withColumn("avg_unit_margin", F.col("gross_margin_amount") / F.col("quantity_sold"))
)

total_rows = forecast_base_df.count()

distinct_grain_rows = (
    forecast_base_df
    .select(FORECAST_GRAIN)
    .distinct()
    .count()
)

duplicate_grain_rows = total_rows - distinct_grain_rows

date_stats = (
    forecast_base_df
    .agg(
        F.min("data_pedido").alias("min_data_pedido"),
        F.max("data_pedido").alias("max_data_pedido"),
        F.countDistinct("data_pedido").alias("distinct_dates"),
        F.countDistinct("produto_id").alias("distinct_products"),
        F.sum("quantity_sold").alias("total_quantity_sold"),
        F.sum("net_sales_amount").alias("total_net_sales_amount"),
        F.avg("quantity_sold").alias("avg_daily_product_quantity"),
        F.max("quantity_sold").alias("max_daily_product_quantity")
    )
    .collect()[0]
)

null_check = (
    forecast_base_df
    .select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in FORECAST_GRAIN + [
            "quantity_sold",
            "net_sales_amount",
            "produto_nome",
            "categoria",
            "marca",
            "status_produto"
        ]
    ])
    .collect()[0]
    .asDict()
)

print(f"Forecast base rows:              {total_rows:,}")
print(f"Distinct grain rows:             {distinct_grain_rows:,}")
print(f"Duplicate grain rows:            {duplicate_grain_rows:,}")
print(f"Min order date:                  {date_stats['min_data_pedido']}")
print(f"Max order date:                  {date_stats['max_data_pedido']}")
print(f"Distinct dates:                  {date_stats['distinct_dates']:,}")
print(f"Distinct products:               {date_stats['distinct_products']:,}")
print(f"Total quantity sold:             {date_stats['total_quantity_sold']:,.2f}")
print(f"Total net sales amount:          {date_stats['total_net_sales_amount']:,.2f}")
print(f"Avg daily product quantity:      {date_stats['avg_daily_product_quantity']:,.2f}")
print(f"Max daily product quantity:      {date_stats['max_daily_product_quantity']:,.2f}")
print(f"Null check:                      {null_check}")

if duplicate_grain_rows != 0:
    raise ValueError("Duplicate grain records found after forecasting aggregation.")

if any(value != 0 for value in null_check.values()):
    raise ValueError("Null values found in critical forecasting columns.")

print("[INFO] Forecasting dataset aggregated and validated successfully.")

display(
    forecast_base_df
    .orderBy("data_pedido", "produto_id")
    .limit(20)
)